# Domain B: Contextual Adaptation and History Dependence

This notebook addresses three questions:
- **B1:** Do responses differ between the first and second presentation of the same block type (context-dependent adaptation)?
- **B2:** Does the context (contrast-varying vs. speed-varying) alter tuning curves?
- **B3:** Do responses change across days (multi-day representational drift)?

**Data:** Zarr multimodal stores with ΔF/F calcium traces (X matrix, cells × trials), stimulus metadata (var), cell-type labels & gene expression (obs).
Context structure: Blocks 0,2 = contrast-context (TF fixed at 1 Hz); Blocks 1,3 = speed-context (contrast fixed at 0.8).

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, wilcoxon, mannwhitneyu, spearmanr, pearsonr, ttest_rel
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr, load_zarr_10hz
from functions.analysis import get_block_responses, compute_adaptation_index, get_trial_position_responses, exp_decay, linear_CKA
from functions.tuning import naka_rushton, normalization_model

sns.set_context('talk')
sns.set_style('whitegrid')

ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

print(f"Total cells: {len(obs)}, Total trials: {X.shape[1]}")

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

# ── Subclass setup ──
SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]
short_order = [SUBCLASS_SHORT[s] for s in present_subclasses]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])
contrasts = np.array([0.05, 0.1, 0.2, 0.4, 0.8])
tfs = np.array([1, 2, 4, 8, 15])

## B1: Block-to-Block Context-Dependent Adaptation

Compare responses to identical stimuli between Block 0 vs Block 2 (contrast context) and Block 1 vs Block 3 (speed context). Compute adaptation indices. Test cell-type specificity.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# B1.1  Compute mean response per cell per stimulus condition in each block
# ══════════════════════════════════════════════════════════════════════

# Contrast-context blocks (0 vs 2): contrast varies, TF = 1 Hz
resp_block0 = get_block_responses(X, var, 0.0, 'contrast', contrasts)
resp_block2 = get_block_responses(X, var, 2.0, 'contrast', contrasts)

# Speed-context blocks (1 vs 3): TF varies, contrast = 0.8
resp_block1 = get_block_responses(X, var, 1.0, 'temporal_frequency', tfs)
resp_block3 = get_block_responses(X, var, 3.0, 'temporal_frequency', tfs)

# ── Collapse over all conditions to get a single adaptation index per cell ──
contrast_conditions = [(ori, c) for ori in orientations for c in contrasts]
speed_conditions = [(ori, tf) for ori in orientations for tf in tfs]

adapt_idx_contrast = compute_adaptation_index(resp_block0, resp_block2, contrast_conditions)
adapt_idx_speed = compute_adaptation_index(resp_block1, resp_block3, speed_conditions)

adapt_df = pd.DataFrame({
    'subclass': obs['subclass_name'].values,
    'subclass_short': obs['subclass_name'].map(SUBCLASS_SHORT).values,
    'mouse_id': obs['mouse_id'].values,
    'adapt_contrast': adapt_idx_contrast,
    'adapt_speed': adapt_idx_speed,
})
adapt_df['adapt_mean'] = adapt_df[['adapt_contrast', 'adapt_speed']].mean(axis=1)

print("Adaptation index statistics:")
print(adapt_df.groupby('subclass_short')[['adapt_contrast', 'adapt_speed']].describe().round(3))

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# B1.2  Statistical tests and visualization of adaptation per cell type
# ══════════════════════════════════════════════════════════════════════

# ── Kruskal-Wallis across subclasses ──
for metric_name, metric_col in [('Contrast-context adapt.', 'adapt_contrast'),
                                 ('Speed-context adapt.', 'adapt_speed')]:
    groups = [adapt_df.loc[adapt_df['subclass'] == s, metric_col].dropna().values
              for s in present_subclasses]
    groups = [g for g in groups if len(g) >= 3]
    if len(groups) >= 2:
        stat, p = kruskal(*groups)
        print(f"{metric_name}: Kruskal-Wallis H={stat:.2f}, p={p:.2e}")

# ── Paired Wilcoxon: is adaptation significantly != 0 per subclass? ──
print("\nOne-sample Wilcoxon (adaptation != 0):")
for sc in present_subclasses:
    for col_name, col in [('Contrast', 'adapt_contrast'), ('Speed', 'adapt_speed')]:
        vals = adapt_df.loc[adapt_df['subclass'] == sc, col].dropna().values
        if len(vals) >= 10:
            stat, p = wilcoxon(vals)
            median = np.median(vals)
            print(f"  {SUBCLASS_SHORT[sc]:10s} {col_name}: median={median:+.4f}, p={p:.2e}")

# ── Visualization: Violin plots of adaptation index ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
short_order = [SUBCLASS_SHORT[s] for s in present_subclasses]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}

for ax, col, title in zip(axes, ['adapt_contrast', 'adapt_speed'],
                            ['Contrast-Context (Block 0 → 2)', 'Speed-Context (Block 1 → 3)']):
    sns.violinplot(data=adapt_df, x='subclass_short', y=col, order=short_order,
                   palette=short_pal, cut=0, inner='box', ax=ax)
    ax.axhline(0, color='k', ls='--', alpha=0.5, lw=1)
    ax.set_title(f'B1: Adaptation Index — {title}', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Adaptation Index\n(positive = weaker response in 2nd block)')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# ── Scatter: Block 0 vs Block 2 mean response, colored by cell type ──
block0_mean = np.nanmean(np.column_stack([resp_block0[c] for c in contrast_conditions]), axis=1)
block2_mean = np.nanmean(np.column_stack([resp_block2[c] for c in contrast_conditions]), axis=1)

fig, axes = plt.subplots(1, len(present_subclasses), figsize=(4*len(present_subclasses), 4), sharex=True, sharey=True)
for ax, sc in zip(axes, present_subclasses):
    mask = obs['subclass_name'].values == sc
    ax.scatter(block0_mean[mask], block2_mean[mask], alpha=0.4, s=15, c=SUBCLASS_COLORS[sc])
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'k--', alpha=0.4, lw=1)
    ax.set_title(SUBCLASS_SHORT[sc], color=SUBCLASS_COLORS[sc], fontweight='bold')
    ax.set_xlabel('Block 0 (1st contrast)')
    if ax == axes[0]:
        ax.set_ylabel('Block 2 (2nd contrast)')
plt.suptitle('B1: Paired Block Responses (Contrast Context)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# B1.3  Within-block trial-position adaptation
# ══════════════════════════════════════════════════════════════════════

n_bins = 5
bin_labels = [f'Bin {i+1}' for i in range(n_bins)]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
block_pairs = [(0, 'Contrast Block 0'), (2, 'Contrast Block 2'),
               (1, 'Speed Block 1'), (3, 'Speed Block 3')]

for ax, (block, title) in zip(axes.flat, block_pairs):
    bin_resp = get_trial_position_responses(X, var, float(block), n_bins=n_bins)
    if bin_resp is None:
        continue
    for sc in present_subclasses:
        mask = obs['subclass_name'].values == sc
        if mask.sum() < 5:
            continue
        mean_trace = np.nanmean(bin_resp[mask], axis=0)
        sem_trace = np.nanstd(bin_resp[mask], axis=0) / np.sqrt(mask.sum())
        x = np.arange(n_bins)
        ax.errorbar(x, mean_trace, yerr=sem_trace, color=SUBCLASS_COLORS[sc],
                    label=SUBCLASS_SHORT[sc], linewidth=2, capsize=3, marker='o')
    ax.set_xticks(range(n_bins))
    ax.set_xticklabels(bin_labels)
    ax.set_xlabel('Trial Position (within block)')
    ax.set_ylabel('Mean ΔF/F')
    ax.set_title(f'B1: {title}', fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
plt.suptitle('B1: Within-Block Adaptation Time Course', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Fit exponential decay to within-block adaptation per subclass ──
print("\nExponential decay fit (within Block 0):")
bin_resp_b0 = get_trial_position_responses(X, var, 0.0, n_bins=10)
for sc in present_subclasses:
    mask = obs['subclass_name'].values == sc
    if mask.sum() < 10:
        continue
    y = np.nanmean(bin_resp_b0[mask], axis=0)
    x = np.arange(len(y))
    try:
        popt, _ = curve_fit(exp_decay, x, y, p0=[y[0]-y[-1], 3, y[-1]], maxfev=5000)
        print(f"  {SUBCLASS_SHORT[sc]:10s}: a={popt[0]:.4f}, tau={popt[1]:.2f} bins, baseline={popt[2]:.4f}")
    except:
        print(f"  {SUBCLASS_SHORT[sc]:10s}: fit failed")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# B1.4  Adaptation heatmap: cells × conditions, sorted by type & magnitude
# ══════════════════════════════════════════════════════════════════════

# Build adaptation matrix per condition (contrast context)
adapt_matrix = np.zeros((adata.n_obs, len(contrast_conditions)))
for ci, cond in enumerate(contrast_conditions):
    r1 = resp_block0[cond]
    r2 = resp_block2[cond]
    denom = np.abs(r1) + np.abs(r2)
    denom[denom < 1e-8] = np.nan
    adapt_matrix[:, ci] = (r1 - r2) / denom

# Sort cells by subclass, then by mean adaptation
sort_df = pd.DataFrame({'subclass': obs['subclass_name'].values,
                         'mean_adapt': np.nanmean(adapt_matrix, axis=1)})
sort_df['subclass_rank'] = sort_df['subclass'].map(
    {s: i for i, s in enumerate(SUBCLASS_ORDER)}).fillna(99).astype(int)
sort_order = sort_df.sort_values(['subclass_rank', 'mean_adapt'], ascending=[True, False]).index

fig, ax = plt.subplots(figsize=(16, 10))
im = ax.imshow(adapt_matrix[sort_order], aspect='auto', cmap='RdBu_r', vmin=-0.5, vmax=0.5,
               interpolation='none')
plt.colorbar(im, ax=ax, label='Adaptation Index')
ax.set_xlabel('Stimulus Condition (orientation × contrast)')
ax.set_ylabel('Cells (sorted by subclass)')

# Mark subclass boundaries
boundaries = []
for sc in present_subclasses:
    sc_mask = obs['subclass_name'].values[sort_order] == sc
    if sc_mask.any():
        boundaries.append((np.where(sc_mask)[0][0], SUBCLASS_SHORT[sc]))
for pos, label in boundaries:
    ax.axhline(pos, color='k', lw=0.5)
    ax.text(-2, pos + 10, label, fontsize=8, fontweight='bold', ha='right', va='center')

ax.set_title('B1: Adaptation Index per Cell × Condition (Contrast Context)', fontweight='bold')
plt.tight_layout()
plt.show()

## B2: Context-Dependent Tuning Curve Modulation

Compare contrast response functions between contrast-context blocks (where contrast is variable) vs the single matched contrast available in speed-context blocks. Similarly for TF tuning. Test for gain control signatures.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# B2.1  Compare CRF between contrast-context blocks and matched contrasts
#       in speed-context blocks
# ══════════════════════════════════════════════════════════════════════

# CRF from contrast-context (blocks 0+2 combined): full 5-level CRF
contrast_blocks = var['stim_block'].isin([0.0, 2.0])
crf_contrast_ctx = np.zeros((adata.n_obs, len(contrasts)))
for i, c in enumerate(contrasts):
    mask = contrast_blocks & (var['contrast'] == c)
    trial_idx = np.where(mask.values)[0]
    if len(trial_idx) > 0:
        crf_contrast_ctx[:, i] = np.nanmean(X[:, trial_idx], axis=1)

# In speed-context blocks (1+3), contrast is fixed at 0.8
# So we get only a single point on the CRF from this context
speed_blocks = var['stim_block'].isin([1.0, 3.0])
mask_08_speed = speed_blocks & (var['contrast'] == 0.8)
resp_08_speed_ctx = np.nanmean(X[:, np.where(mask_08_speed.values)[0]], axis=1)

# Compare: response to contrast=0.8 in contrast-context vs speed-context
resp_08_contrast_ctx = crf_contrast_ctx[:, 4]  # contrast=0.8 is index 4

# ── TF tuning from speed-context (blocks 1+3): full 5-level TF curve ──
tf_speed_ctx = np.zeros((adata.n_obs, len(tfs)))
for i, tf in enumerate(tfs):
    mask = speed_blocks & (var['temporal_frequency'] == tf)
    trial_idx = np.where(mask.values)[0]
    if len(trial_idx) > 0:
        tf_speed_ctx[:, i] = np.nanmean(X[:, trial_idx], axis=1)

# In contrast-context blocks (0+2), TF is fixed at 1 Hz
mask_tf1_contrast = contrast_blocks & (var['temporal_frequency'] == 1.0)
resp_tf1_contrast_ctx = np.nanmean(X[:, np.where(mask_tf1_contrast.values)[0]], axis=1)
resp_tf1_speed_ctx = tf_speed_ctx[:, 0]  # TF=1 is index 0

# ── Naka-Rushton fit to CRF per subclass (from contrast context) ──
fig, axes = plt.subplots(2, len(present_subclasses), figsize=(4*len(present_subclasses), 10))

# Row 1: CRF with overlaid speed-context matched point
for ax, sc in zip(axes[0], present_subclasses):
    mask = obs['subclass_name'].values == sc
    n_sc = mask.sum()
    
    # CRF from contrast context
    mean_crf = np.nanmean(crf_contrast_ctx[mask], axis=0)
    sem_crf = np.nanstd(crf_contrast_ctx[mask], axis=0) / np.sqrt(n_sc)
    ax.errorbar(contrasts, mean_crf, yerr=sem_crf, color=SUBCLASS_COLORS[sc],
                linewidth=2, capsize=3, marker='o', label='Contrast ctx')
    
    # Single point from speed context (c=0.8)
    mean_sp = np.nanmean(resp_08_speed_ctx[mask])
    sem_sp = np.nanstd(resp_08_speed_ctx[mask]) / np.sqrt(n_sc)
    ax.errorbar(0.8, mean_sp, yerr=sem_sp, marker='s', ms=10, color='red',
                zorder=5, label='Speed ctx (c=0.8)')
    
    ax.set_xscale('log')
    ax.set_title(SUBCLASS_SHORT[sc], color=SUBCLASS_COLORS[sc], fontweight='bold')
    ax.set_xlabel('Contrast')
    if ax == axes[0, 0]:
        ax.set_ylabel('Mean ΔF/F')
        ax.legend(fontsize=7)

# Row 2: TF tuning with overlaid contrast-context matched point
for ax, sc in zip(axes[1], present_subclasses):
    mask = obs['subclass_name'].values == sc
    n_sc = mask.sum()
    
    mean_tf = np.nanmean(tf_speed_ctx[mask], axis=0)
    sem_tf = np.nanstd(tf_speed_ctx[mask], axis=0) / np.sqrt(n_sc)
    ax.errorbar(tfs, mean_tf, yerr=sem_tf, color=SUBCLASS_COLORS[sc],
                linewidth=2, capsize=3, marker='o', label='Speed ctx')
    
    mean_ct = np.nanmean(resp_tf1_contrast_ctx[mask])
    sem_ct = np.nanstd(resp_tf1_contrast_ctx[mask]) / np.sqrt(n_sc)
    ax.errorbar(1.0, mean_ct, yerr=sem_ct, marker='s', ms=10, color='blue',
                zorder=5, label='Contrast ctx (TF=1)')
    
    ax.set_xscale('log')
    ax.set_title(SUBCLASS_SHORT[sc], color=SUBCLASS_COLORS[sc], fontweight='bold')
    ax.set_xlabel('Temporal Frequency (Hz)')
    if ax == axes[1, 0]:
        ax.set_ylabel('Mean ΔF/F')
        ax.legend(fontsize=7)

plt.suptitle('B2: Context-Dependent Tuning Curves\n(Top: CRF, Bottom: TF Tuning)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Statistical test: paired context comparison at matched stimulus ──
print("=== Context Modulation: Contrast=0.8 response (contrast-ctx vs speed-ctx) ===")
for sc in present_subclasses:
    mask = obs['subclass_name'].values == sc
    v1 = resp_08_contrast_ctx[mask]
    v2 = resp_08_speed_ctx[mask]
    valid = ~np.isnan(v1) & ~np.isnan(v2)
    if valid.sum() >= 10:
        stat, p = wilcoxon(v1[valid], v2[valid])
        diff = np.mean(v1[valid]) - np.mean(v2[valid])
        print(f"  {SUBCLASS_SHORT[sc]:10s}: Δ={diff:+.4f}, Wilcoxon p={p:.2e}")

print("\n=== Context Modulation: TF=1 response (speed-ctx vs contrast-ctx) ===")
for sc in present_subclasses:
    mask = obs['subclass_name'].values == sc
    v1 = resp_tf1_speed_ctx[mask]
    v2 = resp_tf1_contrast_ctx[mask]
    valid = ~np.isnan(v1) & ~np.isnan(v2)
    if valid.sum() >= 10:
        stat, p = wilcoxon(v1[valid], v2[valid])
        diff = np.mean(v1[valid]) - np.mean(v2[valid])
        print(f"  {SUBCLASS_SHORT[sc]:10s}: Δ={diff:+.4f}, Wilcoxon p={p:.2e}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# B2.2  Normalization / Gain Control Model Fit
#       Fit the Carandini-Heeger divisive normalization model to CRFs
#       Compare normalization pool parameters across cell types
# ══════════════════════════════════════════════════════════════════════

# Fit per cell for contrast-context CRFs
norm_params = {'C50_norm': [], 'Rmax_norm': [], 'n_norm': []}
for i in range(adata.n_obs):
    resp = crf_contrast_ctx[i]
    if np.ptp(resp) < 0.01 or np.any(np.isnan(resp)):
        norm_params['C50_norm'].append(np.nan)
        norm_params['Rmax_norm'].append(np.nan)
        norm_params['n_norm'].append(np.nan)
        continue
    try:
        popt, _ = curve_fit(normalization_model, contrasts, resp,
                           p0=[np.max(resp), 0.3, 2.0],
                           bounds=([0, 0.01, 0.1], [20, 2.0, 10]),
                           maxfev=5000)
        norm_params['C50_norm'].append(popt[1])
        norm_params['Rmax_norm'].append(popt[0])
        norm_params['n_norm'].append(popt[2])
    except:
        norm_params['C50_norm'].append(np.nan)
        norm_params['Rmax_norm'].append(np.nan)
        norm_params['n_norm'].append(np.nan)

adapt_df['C50_norm'] = norm_params['C50_norm']
adapt_df['Rmax_norm'] = norm_params['Rmax_norm']
adapt_df['n_norm'] = norm_params['n_norm']

# ── Visualization: Normalization parameters by subclass ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, param in zip(axes, ['C50_norm', 'Rmax_norm', 'n_norm']):
    sns.violinplot(data=adapt_df[adapt_df['subclass'].isin(present_subclasses)],
                   x='subclass_short', y=param, order=short_order,
                   palette=short_pal, cut=0, inner='box', ax=ax)
    ax.set_title(f'B2: {param}', fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('B2: Normalization Model Parameters by Subclass', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## B3: Multi-Day Representational Drift

Track how responses to identical stimuli change across the 3 recording days. Compute stability metrics per cell type. Analyze population-level drift via PCA.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# B3.1  Cross-day response stability analysis
# ══════════════════════════════════════════════════════════════════════

# The var dataframe has a 'day' column (1, 2, 3) for the 3 recording days
days = sorted(var['day'].unique())
print(f"Days in dataset: {days}")

# Compute mean response per cell per stimulus condition per day
# Use all blocks combined; conditions = orientation × contrast × TF (use stim_index_block)
# Simplify: use orientation tuning vector per day (contrast-context blocks, collapse contrast)

day_ori_responses = {}
for day in days:
    day_mask = var['day'] == day
    contrast_day = day_mask & var['stim_block'].isin([0.0, 2.0])
    resp = np.zeros((adata.n_obs, len(orientations)))
    for i, ori in enumerate(orientations):
        mask = contrast_day & (var['orientation'] == ori)
        trial_idx = np.where(mask.values)[0]
        if len(trial_idx) > 0:
            resp[:, i] = np.nanmean(X[:, trial_idx], axis=1)
    day_ori_responses[day] = resp

# ── Cross-day correlation per cell ──
from itertools import combinations

day_pairs = list(combinations(days, 2))
stability_records = []

for cell_i in range(adata.n_obs):
    for d1, d2 in day_pairs:
        v1 = day_ori_responses[d1][cell_i]
        v2 = day_ori_responses[d2][cell_i]
        if np.std(v1) > 1e-6 and np.std(v2) > 1e-6:
            r, _ = pearsonr(v1, v2)
        else:
            r = np.nan
        stability_records.append({
            'cell': cell_i,
            'day_pair': f'{d1}-{d2}',
            'correlation': r,
            'subclass': obs['subclass_name'].iloc[cell_i],
            'subclass_short': SUBCLASS_SHORT.get(obs['subclass_name'].iloc[cell_i], 'Other'),
        })

stab_df = pd.DataFrame(stability_records)

# ── Visualization: Cross-day stability by subclass ──
fig, axes = plt.subplots(1, len(day_pairs), figsize=(6*len(day_pairs), 5))
if len(day_pairs) == 1:
    axes = [axes]

for ax, dp in zip(axes, [f'{d1}-{d2}' for d1, d2 in day_pairs]):
    sub = stab_df[stab_df['day_pair'] == dp]
    sns.violinplot(data=sub[sub['subclass'].isin(present_subclasses)],
                   x='subclass_short', y='correlation', order=short_order,
                   palette=short_pal, cut=0, inner='box', ax=ax)
    ax.axhline(0, color='k', ls='--', alpha=0.4)
    ax.set_title(f'B3: Day {dp} Stability', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Pearson r (orientation vector)')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('B3: Cross-Day Tuning Stability by Subclass', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Kruskal-Wallis across subclasses for stability ──
mean_stab = stab_df.groupby(['cell', 'subclass'])['correlation'].mean().reset_index()
groups = [mean_stab.loc[mean_stab['subclass'] == s, 'correlation'].dropna().values
          for s in present_subclasses]
groups = [g for g in groups if len(g) >= 3]
if len(groups) >= 2:
    stat, p = kruskal(*groups)
    print(f"Cross-day stability Kruskal-Wallis: H={stat:.2f}, p={p:.2e}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# B3.2  Population-level drift via PCA
# ══════════════════════════════════════════════════════════════════════
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# Build population response matrix per day: cells × orientations
# Then apply PCA to the combined space
all_day_resp = np.vstack([day_ori_responses[d] for d in days])  # (3*n_cells) × 8
day_labels = np.concatenate([[d]*adata.n_obs for d in days])

pca = PCA(n_components=3)
pca_coords = pca.fit_transform(all_day_resp)

fig = plt.figure(figsize=(14, 5))

# 2D PCA colored by day
ax1 = fig.add_subplot(131)
for d in days:
    mask = day_labels == d
    ax1.scatter(pca_coords[mask, 0], pca_coords[mask, 1], alpha=0.3, s=10, label=f'Day {d}')
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax1.legend()
ax1.set_title('B3: Population PCA by Day')

# Track mean position per day per subclass
ax2 = fig.add_subplot(132)
for sc in present_subclasses:
    sc_mask_base = obs['subclass_name'].values == sc
    centroids = []
    for d in days:
        d_mask = day_labels == d
        combined_mask = np.zeros(len(day_labels), dtype=bool)
        # This subclass in this day
        start = (d - days[0]) * adata.n_obs
        end = start + adata.n_obs
        combined_mask[start:end] = sc_mask_base
        combined_mask = combined_mask & d_mask
        if combined_mask.sum() > 0:
            centroids.append(pca_coords[combined_mask].mean(axis=0))
    if len(centroids) == len(days):
        centroids = np.array(centroids)
        ax2.plot(centroids[:, 0], centroids[:, 1], 'o-', color=SUBCLASS_COLORS[sc],
                label=SUBCLASS_SHORT[sc], linewidth=2, markersize=8)
        # Arrow from day 1 to day 3
        ax2.annotate('', xy=centroids[-1, :2], xytext=centroids[0, :2],
                    arrowprops=dict(arrowstyle='->', color=SUBCLASS_COLORS[sc], lw=1.5))
ax2.set_xlabel(f'PC1')
ax2.set_ylabel(f'PC2')
ax2.legend(fontsize=7)
ax2.set_title('B3: Subclass Centroid Drift')

# Euclidean drift distance per subclass
ax3 = fig.add_subplot(133)
drift_data = []
for sc in present_subclasses:
    sc_mask_base = obs['subclass_name'].values == sc
    for cell_i in np.where(sc_mask_base)[0]:
        coords = []
        for d in days:
            idx = (d - days[0]) * adata.n_obs + cell_i
            coords.append(pca_coords[idx])
        coords = np.array(coords)
        # Total drift = sum of day-to-day euclidean distances
        total_drift = sum(np.linalg.norm(coords[j+1] - coords[j]) for j in range(len(days)-1))
        drift_data.append({'subclass_short': SUBCLASS_SHORT[sc], 'drift': total_drift})

drift_df = pd.DataFrame(drift_data)
sns.boxplot(data=drift_df, x='subclass_short', y='drift', order=short_order,
            palette=short_pal, ax=ax3)
ax3.set_title('B3: Total PCA Drift by Subclass')
ax3.set_xlabel('')
ax3.set_ylabel('Total PCA drift (Days 1→3)')
ax3.tick_params(axis='x', rotation=45)

plt.suptitle('B3: Multi-Day Representational Drift', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── CKA: compare population RDMs across days ──
print("\n=== CKA between days (population orientation responses) ===")
for d1, d2 in day_pairs:
    cka = linear_CKA(day_ori_responses[d1], day_ori_responses[d2])
    print(f"  Day {d1} vs Day {d2}: CKA = {cka:.4f}")

## B4: Within-Trial Temporal Adaptation (10 Hz ΔF/F)

Using 10 Hz trial-resolved traces, we examine how the temporal shape of responses changes:
- **B4.1** How does the PSTH shape evolve from the first to the last trial within a block? (within-block habituation)
- **B4.2** Does the transient/sustained ratio shift between contrast-context vs speed-context blocks?

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# B4.1  Load 10 Hz data & examine within-block PSTH evolution
# ══════════════════════════════════════════════════════════════════════

# Use one representative mouse (largest)
mouse_pick = obs['mouse_id'].value_counts().idxmax()
pk = load_zarr_10hz(mouse_pick, zarr_dir=ZARR_DIR)
dff_10hz = pk['dff']            # (n_trials, 41, n_cells)
ti_10hz = pk['trial_info']
time_rel = pk['time_rel']       # (41,) from -1s to 3s
uids_10hz = pk['unique_ids']
n_trials_pk, n_tp, n_cells_pk = dff_10hz.shape

# Match cells to obs for subclass labels
obs_mouse = obs[obs['mouse_id'] == mouse_pick]
cell_sc_map = {}
for ci, uid in enumerate(uids_10hz):
    match = obs_mouse[obs_mouse['unique_id'] == uid]
    if len(match) > 0:
        cell_sc_map[ci] = match.iloc[0]['subclass_name']

matched_cells = sorted(cell_sc_map.keys())
cell_subclasses = np.array([cell_sc_map[ci] for ci in matched_cells])

# Identify block boundaries in trial_info
non_gray = ~ti_10hz['gray_screen'].astype(bool)
block_ids = ti_10hz.loc[non_gray.values, 'stim_block'].values
grating_trial_idx = np.where(non_gray.values)[0]

print(f"Mouse {mouse_pick}: {n_cells_pk} cells, {n_trials_pk} total trials, {len(grating_trial_idx)} grating trials")
print(f"Matched {len(matched_cells)} cells to taxonomy")

# ── Within-block temporal evolution ──
# Split each block's grating trials into early (first 20%) vs late (last 20%)
stim_idx = (time_rel >= 0) & (time_rel <= 2.0)
early_idx_t = (time_rel >= 0) & (time_rel <= 0.5)
late_idx_t = (time_rel >= 1.5) & (time_rel <= 2.0)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
block_names = {0: 'Block 0 (Contrast-ctx)', 1: 'Block 1 (Speed-ctx)',
               2: 'Block 2 (Contrast-ctx)', 3: 'Block 3 (Speed-ctx)'}

for bi, block in enumerate([0, 1, 2, 3]):
    ax = axes.flat[bi]
    block_trial_mask = block_ids == block
    block_trials = grating_trial_idx[block_trial_mask]
    n_bt = len(block_trials)
    n_fifth = max(1, n_bt // 5)
    
    early_trials = block_trials[:n_fifth]
    late_trials = block_trials[-n_fifth:]
    
    for sc in present_subclasses:
        sc_cells = [matched_cells[i] for i in range(len(matched_cells)) if cell_subclasses[i] == sc]
        if len(sc_cells) < 3:
            continue
        dff_e = np.nanmean(dff_10hz[early_trials][:, :, sc_cells], axis=(0, 2))
        dff_l = np.nanmean(dff_10hz[late_trials][:, :, sc_cells], axis=(0, 2))
        ax.plot(time_rel, dff_e, color=SUBCLASS_COLORS[sc], linewidth=2, label=f'{SUBCLASS_SHORT[sc]} early')
        ax.plot(time_rel, dff_l, color=SUBCLASS_COLORS[sc], linewidth=2, ls='--', label=f'{SUBCLASS_SHORT[sc]} late')
    
    ax.axvline(0, color='k', ls='--', alpha=0.3)
    ax.set_title(block_names[block], fontweight='bold')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('ΔF/F')
    if bi == 0:
        ax.legend(fontsize=6, ncol=2)

plt.suptitle('B4: Within-Block PSTH Adaptation (Early vs Late Trials)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Per-subclass: PSTH shape comparison (contrast vs speed context) ──
present_sc_b = [s for s in present_subclasses if sum(1 for c in matched_cells if cell_sc_map.get(c) == s) >= 3]
fig, axes = plt.subplots(len(present_sc_b), 2, figsize=(14, 4*len(present_sc_b)))
if len(present_sc_b) == 1:
    axes = axes.reshape(1, -1)

for ri, sc in enumerate(present_sc_b):
    sc_cells = [matched_cells[i] for i in range(len(matched_cells)) if cell_subclasses[i] == sc]
    if len(sc_cells) < 3:
        continue
    
    for ci_block, (block_label, block_val) in enumerate([('Contrast-ctx (0,2)', [0, 2]),
                                                          ('Speed-ctx (1,3)', [1, 3])]):
        ax = axes[ri, ci_block]
        bm = np.isin(block_ids, block_val)
        bt = grating_trial_idx[bm]
        n_bt = len(bt)
        n_fifth = max(1, n_bt // 5)
        
        dff_e = np.nanmean(dff_10hz[bt[:n_fifth]][:, :, sc_cells], axis=(0, 2))
        dff_l = np.nanmean(dff_10hz[bt[-n_fifth:]][:, :, sc_cells], axis=(0, 2))
        
        ax.plot(time_rel, dff_e, color=SUBCLASS_COLORS[sc], linewidth=2, label='Early')
        ax.plot(time_rel, dff_l, color=SUBCLASS_COLORS[sc], linewidth=2, ls='--', label='Late')
        ax.axvline(0, color='k', ls='--', alpha=0.3)
        ax.set_title(f'{SUBCLASS_SHORT[sc]} — {block_label}', fontweight='bold',
                     color=SUBCLASS_COLORS[sc], fontsize=10)
        ax.set_ylabel('ΔF/F')
        ax.legend(fontsize=7)

plt.suptitle('B4: Within-Block PSTH Adaptation by Subclass', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# B4.2  Transient/sustained ratio shifts between block contexts
# ══════════════════════════════════════════════════════════════════════

# Per-cell TSI in contrast-context vs speed-context blocks
tsi_by_context = {'contrast': [], 'speed': [], 'subclass': [], 'cell_idx': []}

for ci_idx, ci in enumerate(matched_cells):
    for ctx_name, ctx_blocks in [('contrast', [0, 2]), ('speed', [1, 3])]:
        ctx_mask = np.isin(block_ids, ctx_blocks)
        ctx_trials = grating_trial_idx[ctx_mask]
        if len(ctx_trials) < 10:
            continue
        
        mean_psth = np.nanmean(dff_10hz[ctx_trials, :, ci], axis=0)  # (41,)
        early_r = np.nanmean(mean_psth[early_idx_t])
        late_r = np.nanmean(mean_psth[late_idx_t])
        tsi_val = (early_r - late_r) / (np.abs(early_r) + np.abs(late_r) + 1e-8)
        
        tsi_by_context[ctx_name].append(tsi_val)
        if ctx_name == 'contrast':
            tsi_by_context['subclass'].append(cell_subclasses[ci_idx])
            tsi_by_context['cell_idx'].append(ci)

tsi_ctx_df = pd.DataFrame({
    'TSI_contrast': tsi_by_context['contrast'],
    'TSI_speed': tsi_by_context['speed'],
    'subclass': tsi_by_context['subclass'],
    'subclass_short': [SUBCLASS_SHORT.get(s, s) for s in tsi_by_context['subclass']],
})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Scatter: TSI contrast vs TSI speed ──
ax = axes[0]
for sc in present_sc_b:
    m = tsi_ctx_df['subclass'] == sc
    ax.scatter(tsi_ctx_df.loc[m, 'TSI_contrast'], tsi_ctx_df.loc[m, 'TSI_speed'],
               alpha=0.4, s=15, color=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc])
lims = [-1.5, 1.5]
ax.plot(lims, lims, 'k--', alpha=0.3)
ax.set_xlabel('TSI (Contrast Context)')
ax.set_ylabel('TSI (Speed Context)')
ax.set_title('B4: Transient/Sustained Index by Context', fontweight='bold')
ax.legend(fontsize=7)
ax.set_xlim(lims); ax.set_ylim(lims)

# ── Paired difference: TSI_contrast - TSI_speed ──
ax = axes[1]
tsi_ctx_df['TSI_diff'] = tsi_ctx_df['TSI_contrast'] - tsi_ctx_df['TSI_speed']
short_order_b = [SUBCLASS_SHORT[s] for s in present_sc_b]
short_pal_b = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}
sns.violinplot(data=tsi_ctx_df, x='subclass_short', y='TSI_diff',
               order=short_order_b, palette=short_pal_b, cut=0, inner='box', ax=ax)
ax.axhline(0, color='k', ls='--', alpha=0.5)
ax.set_ylabel('ΔTSI (Contrast - Speed)')
ax.set_title('B4: Context-Dependent TSI Shift', fontweight='bold')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=45)

# ── Within-trial time course for contrast vs speed blocks (population mean) ──
ax = axes[2]
for ctx_name, ctx_blocks, color in [('Contrast-ctx', [0, 2], 'blue'), ('Speed-ctx', [1, 3], 'orange')]:
    ctx_mask = np.isin(block_ids, ctx_blocks)
    ctx_trials = grating_trial_idx[ctx_mask]
    mean_p = np.nanmean(dff_10hz[ctx_trials][:, :, matched_cells], axis=(0, 2))
    sem_p = np.nanstd(np.nanmean(dff_10hz[ctx_trials][:, :, matched_cells], axis=2), axis=0) / np.sqrt(len(ctx_trials))
    ax.fill_between(time_rel, mean_p - sem_p, mean_p + sem_p, alpha=0.2, color=color)
    ax.plot(time_rel, mean_p, color=color, linewidth=2, label=ctx_name)
ax.axvline(0, color='k', ls='--', alpha=0.4)
ax.axvline(2.0, color='k', ls=':', alpha=0.4)
ax.set_xlabel('Time (s)')
ax.set_ylabel('ΔF/F')
ax.set_title('B4: Population PSTH by Block Context', fontweight='bold')
ax.legend()

plt.suptitle('B4: Context Effects on Temporal Response Shape', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Stats: paired Wilcoxon on TSI_contrast vs TSI_speed per subclass ──
from scipy.stats import wilcoxon
print("\n=== Paired Wilcoxon: TSI contrast vs speed context ===")
for sc in present_sc_b:
    m = tsi_ctx_df['subclass'] == sc
    c_vals = tsi_ctx_df.loc[m, 'TSI_contrast'].values
    s_vals = tsi_ctx_df.loc[m, 'TSI_speed'].values
    valid = ~np.isnan(c_vals) & ~np.isnan(s_vals)
    if valid.sum() >= 10:
        stat, p = wilcoxon(c_vals[valid], s_vals[valid])
        print(f"  {SUBCLASS_SHORT[sc]:10s}: W={stat:.0f}, p={p:.2e}, n={valid.sum()}")